In [1]:
# IMPORT LIBRARIES

In [2]:
import pandas as pd
import sqlite3

In [3]:
# CONNECT TO DATABASE

In [4]:
conn = sqlite3.connect('ipl_data.db')
print("Database connected successfully")

Database connected successfully


In [5]:
# FUNCTION: GET ANY TEAM'S OVERALL RECORD

In [6]:
def get_team_record(team_name):
    Query = """
    SELECT ? as team,
           COUNT(*) as total_matches,
           COUNT(CASE WHEN winner LIKE ? THEN 1 END) as wins,
            COUNT(*) - COUNT(CASE WHEN winner LIKE ? THEN 1 END) AS losses,
           ROUND(COUNT(CASE WHEN winner LIKE ? THEN 1 END) * 100.0 / COUNT(*) ,1) as win_pct
    FROM matches
    WHERE team1 LIKE ? or team2 LIKE ?
    """
    result = pd.read_sql_query(Query, conn, params=(team_name, team_name, team_name, team_name, team_name, team_name))
    return result

print("=== RCB ===")
print(get_team_record('Royal Challengers%'))

print("=== CSK ===")
print(get_team_record('Chennai%'))

print("\n=== Mumbai Indians ===")
print(get_team_record('Mumbai Indians%'))

=== RCB ===
                 team  total_matches  wins  losses  win_pct
0  Royal Challengers%            255   123     132     48.2
=== CSK ===
       team  total_matches  wins  losses  win_pct
0  Chennai%            238   138     100     58.0

=== Mumbai Indians ===
              team  total_matches  wins  losses  win_pct
0  Mumbai Indians%            261   144     117     55.2


In [7]:
# FUNCTION: ANY TEAM'S SEASON PERFORMANCE

In [8]:
def get_team_season_performance(team_name):
    Query = """
    SELECT 
           season,
           COUNT(*) as total_matches,
           COUNT(CASE WHEN winner LIKE ? THEN 1 END) as wins,
           ROUND(COUNT(CASE WHEN winner LIKE ? THEN 1 END) * 100.0 / COUNT(*) ,1) as win_pct
    FROM matches
    WHERE team1 LIKE ? or team2 LIKE ?
    GROUP BY season
    """
    result = pd.read_sql_query(Query,conn,params = (team_name, team_name, team_name, team_name))
    return result

print("=== RCB Season Performance ===")
print(get_team_season_performance('Royal Challengers%'))

print("\n=== KKR Season Performance ===")
print(get_team_season_performance('Kolkata Knight%'))



=== RCB Season Performance ===
     season  total_matches  wins  win_pct
0   2007/08             14     4     28.6
1      2009             16     9     56.3
2   2009/10             16     8     50.0
3      2011             16    10     62.5
4      2012             15     8     53.3
5      2013             16     9     56.3
6      2014             14     5     35.7
7      2015             16     8     50.0
8      2016             16     9     56.3
9      2017             13     3     23.1
10     2018             14     6     42.9
11     2019             14     5     35.7
12  2020/21             15     7     46.7
13     2021             15     9     60.0
14     2022             16     9     56.3
15     2023             14     7     50.0
16     2024             15     7     46.7

=== KKR Season Performance ===
     season  total_matches  wins  win_pct
0   2007/08             13     6     46.2
1      2009             13     3     23.1
2   2009/10             14     7     50.0
3      2011  

In [9]:
# FUNCTION: ANY BATSMAN'S CAREER STATS

In [10]:
def get_batsman_stats(batter_name):
    Query = """
    WITH batter_match_totals AS (
            SELECT 
                match_id,
                batter,
                SUM(batsman_runs) AS match_runs
            FROM deliveries
            WHERE batter = ?
            GROUP BY match_id
        )
        SELECT 
            batter,
            SUM(match_runs) AS total_runs,
            COUNT(match_id) AS matches_played,
            ROUND(SUM(match_runs) * 1.0 / COUNT(match_id), 1) AS avg_runs_per_match,
            MAX(match_runs) AS highest_score
        FROM batter_match_totals
        GROUP BY batter;
    """
    result = pd.read_sql_query(Query,conn,params = ((batter_name,)))
    return result

print("=== V Kohli ===")
print(get_batsman_stats('V Kohli'))

print("\n=== AB de Villiers ===")
print(get_batsman_stats('AB de Villiers'))

print("\n=== MS DHONI ===")
print(get_batsman_stats('MS Dhoni'))

=== V Kohli ===
    batter  total_runs  matches_played  avg_runs_per_match  highest_score
0  V Kohli        8014             244                32.8            113

=== AB de Villiers ===
           batter  total_runs  matches_played  avg_runs_per_match  \
0  AB de Villiers        5181             170                30.5   

   highest_score  
0            133  

=== MS DHONI ===
     batter  total_runs  matches_played  avg_runs_per_match  highest_score
0  MS Dhoni        5243             228                23.0             84


In [11]:
# FUNCTION: COMPARE TWO TEAMS SIDE-BY-SIDE

In [12]:

def compare_teams(team1, team2):
    query = """
    SELECT 
        ? AS team,
        COUNT(*) AS total_matches,
        COUNT(CASE WHEN winner LIKE ? THEN 1 END) AS wins,
        ROUND(COUNT(CASE WHEN winner LIKE ? THEN 1 END) * 100.0 / COUNT(*), 1) AS win_pct
    FROM matches
    WHERE team1 LIKE ?
       OR team2 LIKE ?
    
    UNION ALL
    
    SELECT 
        ? AS team,
        COUNT(*) AS total_matches,
        COUNT(CASE WHEN winner = ? THEN 1 END) AS wins,
        ROUND(COUNT(CASE WHEN winner = ? THEN 1 END) * 100.0 / COUNT(*), 1) AS win_pct
    FROM matches
    WHERE team1 = ?
       OR team2 = ?
    """
    
    result = pd.read_sql_query(query, conn, 
        params=(team1, team1, team1, team1, team1,
                team2, team2, team2, team2, team2))
    return result

# Test
print("=== RCB vs CSK ===")
print(compare_teams('Royal Challengers%', 'Chennai Super Kings'))

print("\n=== MI vs KKR ===")
print(compare_teams('Mumbai Indians', 'Kolkata Knight Riders'))

=== RCB vs CSK ===
                  team  total_matches  wins  win_pct
0   Royal Challengers%            255   123     48.2
1  Chennai Super Kings            238   138     58.0

=== MI vs KKR ===
                    team  total_matches  wins  win_pct
0         Mumbai Indians            261   144     55.2
1  Kolkata Knight Riders            251   131     52.2


In [13]:
# FUNCTION: FULL SEASON SUMMARY

In [14]:

def season_summary(year):
    query1 = """
    SELECT 
        ? AS season,
        COUNT(DISTINCT id) AS total_matches,
        (
            SELECT winner
            FROM matches
            WHERE season = ?
              AND winner IS NOT NULL
            GROUP BY winner
            ORDER BY COUNT(*) DESC
            LIMIT 1
        ) AS most_wins_team,
        (
            SELECT winner
            FROM matches
            WHERE season = ?
              AND winner IS NOT NULL
            GROUP BY winner
            ORDER BY COUNT(*) ASC
            LIMIT 1
        ) AS least_wins_team
    FROM matches
    WHERE season = ?
    """
    
    result1 = pd.read_sql_query(query1, conn, params=(year, year, year, year))
    
    query2 = """
    SELECT 
        MAX(match_runs) AS highest_team_score
    FROM (
        SELECT 
            match_id,
            batting_team,
            SUM(total_runs) AS match_runs
        FROM deliveries d
        JOIN matches m ON d.match_id = m.id
        WHERE m.season = ?
        GROUP BY match_id, batting_team
    )
    """
    
    result2 = pd.read_sql_query(query2, conn, params=(year,))
    
    result1['highest_team_score'] = result2['highest_team_score'][0]
    return result1

# Test
print("=== 2016 Season ===")
print(season_summary('2016'))

print("\n=== 2024 Season ===")
print(season_summary('2024'))

=== 2016 Season ===
  season  total_matches       most_wins_team  least_wins_team  \
0   2016             60  Sunrisers Hyderabad  Kings XI Punjab   

   highest_team_score  
0                 248  

=== 2024 Season ===
  season  total_matches         most_wins_team least_wins_team  \
0   2024             71  Kolkata Knight Riders  Mumbai Indians   

   highest_team_score  
0                 287  


In [15]:
# CLOSE CONNECTION

In [16]:
conn.close()
print("Connection closed.")

Connection closed.


## Day 9 Summary: Parameterized Queries

 Completed
- Created get_team_record() — works for any IPL team (tested RCB, MI, CSK)
- Created get_team_seasons() — season-by-season win% for any team
- Created get_batsman_stats() — career stats for any player (Kohli, ABD, Gayle)
- Created compare_teams() — side-by-side comparison using UNION ALL
- Created season_summary() — full season analysis with subqueries

 Key Skills Learned
- ? placeholders for safe parameter passing in SQL
- params=() tuple in pd.read_sql_query()
- Writing team-agnostic, player-agnostic functions
- UNION ALL for combining multiple query results
- Subqueries within parameterized queries

 Next
Day 10 — Window Functions: ROW_NUMBER, RANK, DENSE_RANK, and running totals